# Event classification & departure target — Lausanne residential mutations

**Pipeline notebook 3 (Phase 2).** Classifies the 31 `CODE_MUTATION` values along two
axes — the event *family/nature* and its signed *effect on household size* — then
defines the binary departure target on top of the `CODETA_C` lifecycle state.

Design principles carried from the dataset's operating rules:

- **`CODETA_C` is a *sticky* lifecycle state** (`ArriveeProvisoire → Actif →
  DepartAnticipe → Inactif`) propagating forward. A resident's *terminal* state
  summarises the outcome and automatically nets out cancellations (`Suppression*`
  revert the state) and return migration.
- **Out-of-commune moves are not departures.** `MutationDemenagementHorsCommune`,
  hospitalisation and detention keep the resident `Actif` (temporary absence).
- **Birth and death are demographic boundaries of household composition** (+1 / −1),
  distinct in nature from migratory arrivals and departures (which also change size).


## 0. Load the audited master

Loads `masterfile_audited` from notebook 2. If it was saved as compressed CSV the
dates come back as **text in one of two dialects**: ISO 8601 (what `to_csv` writes
for datetime columns — the parquet-fallback case) or the raw Swiss registry format
(`%d.%m.%Y %H:%M`). Parsing ISO text with the Swiss format under `errors="coerce"`
would silently turn every date into `NaT`, so the guard tries ISO first and falls
back to the Swiss format only if ISO clearly fails.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data")
AUD_PQ   = DATA_DIR / "masterfile_audited.parquet"
AUD_GZ   = DATA_DIR / "masterfile_audited.csv.gz"

if AUD_PQ.exists():
    clean = pd.read_parquet(AUD_PQ)
elif AUD_GZ.exists():
    clean = pd.read_csv(AUD_GZ, dtype=str, low_memory=False)
else:
    raise FileNotFoundError("Run notebook 2 first (masterfile_audited).")

DATE_FMT  = "%d.%m.%Y %H:%M"
DATE_COLS = ["DATE_EFFECTIVE", "MUTATION_DATE", "DATDEP", "DDECES", "DATNAIS", "DATARR"]
for c in DATE_COLS:
    if c in clean.columns and not pd.api.types.is_datetime64_any_dtype(clean[c]):
        parsed = pd.to_datetime(clean[c], format="ISO8601", errors="coerce")
        if parsed.isna().mean() > 0.5:                      # not ISO -> Swiss registry format
            parsed = pd.to_datetime(clean[c], format=DATE_FMT, errors="coerce")
        clean[c] = parsed

print(f"Loaded: {len(clean):,} rows x {clean.shape[1]} columns")

Loaded: 863,577 rows x 68 columns


## 1. Event classification — family, nature, household-size effect

Each of the 31 codes maps to one **family** via an explicit dictionary (substring
rules are unsafe — "menage" is a substring of "demenagement"). Births are split from
migratory arrivals. Two derived columns are added:

- `nature_evenement` — `demographique` (birth, death), `migratoire` (arrival,
  departure, announced departure), `residentiel` (internal move), `menage`
  (regroup/split), `absence` (out-of-commune, hospital, detention), `administratif`
  (corrections), `annulation` (suppressions).
- `effet_taille_menage` — signed effect on household size: `+1` birth, `+1` arrival,
  `−1` death, `−1` departure. **`MutationDepartNonConfirme` is also `−1`** via a
  code-level override: the move has already happened, only the inter-commune
  confirmation is pending — consistent with the target rule that counts it as a
  departure immediately. `MutationDepartAnticipe` stays `0` (the resident has not
  left yet). Household regroup/split are left at `0` because their net effect is
  not a fixed ±1.

Any unmapped code **halts the notebook** (`assert`): the schema is never assumed
stable across batches, so a 32nd code in a future download must fail loudly, not
pass silently.

In [2]:
FAMILY = {}
def _add(fam, codes):
    for c in codes:
        FAMILY[c] = fam

_add("departure",           ["MutationDepartDefinitif"])
_add("departure_announced", ["MutationDepartAnticipe", "MutationDepartNonConfirme"])
_add("arrival",             ["MutationArriveeProvisoireEnDefinitive", "MutationArriveeDefinitive"])
_add("birth",               ["MutationNaissance"])
_add("death",               ["MutationDeces"])
_add("internal_move",       ["MutationDemenagement"])
_add("temporary_absence",   ["MutationDemenagementHorsCommune",
                             "MutationDetention", "MutationHospitalisation"])
_add("household",           ["MutationRegroupementMenage", "MutationDegroupementMenage",
                             "CorrectionRegroupementMenage", "CorrectionDegroupementMenage"])
_add("correction",          ["CorrectionAdresseEffective", "CorrectionSejour",
                             "MutationAdresseAdministrative", "CorrectionAdresse"])
_add("suppression",         ["SuppressionDepartDefinitif", "SuppressionArriveeProvisoire",
                             "SuppressionDemenagement", "SuppressionDepartNonConfirme",
                             "SuppressionDepartAnticipe", "SuppressionDossier",
                             "SuppressionArriveeDefinitive", "SuppressionAdresseAdministrative",
                             "SuppressionDemenagementHorsCommune", "SuppressionDeces",
                             "SuppressionDetention", "SuppressionHospitalisation"])

NATURE = {
    "birth": "demographique", "death": "demographique",
    "arrival": "migratoire", "departure": "migratoire", "departure_announced": "migratoire",
    "internal_move": "residentiel", "household": "menage",
    "temporary_absence": "absence", "correction": "administratif", "suppression": "annulation",
}
EFFET_FAMILY = {"birth": 1, "arrival": 1, "death": -1, "departure": -1}
# Code-level override: the non-confirmed departure has physically happened already.
EFFET_CODE   = {"MutationDepartNonConfirme": -1}

clean["famille_evenement"]   = clean["CODE_MUTATION"].map(FAMILY)
clean["nature_evenement"]    = clean["famille_evenement"].map(NATURE)
clean["effet_taille_menage"] = (clean["CODE_MUTATION"].map(EFFET_CODE)
                                .fillna(clean["famille_evenement"].map(EFFET_FAMILY))
                                .fillna(0).astype("int8"))

unmapped = (clean.loc[clean["famille_evenement"].isna() & clean["CODE_MUTATION"].notna(),
                      "CODE_MUTATION"].unique())
assert len(unmapped) == 0, f"Unmapped CODE_MUTATION values: {list(unmapped)}"
print("All CODE_MUTATION values mapped.")
print("\nEvent-family distribution:")
print(clean["famille_evenement"].value_counts(dropna=False).to_string())
print("\nNature x household-size effect:")
print(clean.groupby(["nature_evenement", "effet_taille_menage"]).size().to_string())

All CODE_MUTATION values mapped.

Event-family distribution:
famille_evenement
departure              199440
arrival                196153
internal_move          141751
departure_announced    117868
correction              86094
household               61415
suppression             29224
birth                   18440
death                   11865
temporary_absence        1327

Nature x household-size effect:
nature_evenement  effet_taille_menage
absence            0                       1327
administratif      0                      86094
annulation         0                      29224
demographique     -1                      11865
                   1                      18440
menage             0                      61415
migratoire        -1                     243102
                   0                      74206
                   1                     196153
residentiel        0                     141751


## 2. Terminal lifecycle state per individual

`CODETA_C` snapshots the registry state **at entry time**: the administratively most
recent row (max `MUTATION_DATE`) carries the current truth, even when it retro-dates
an older `DATE_EFFECTIVE` — e.g. a suppression entered in May 2026 that cancels a
December 2025 move. The terminal state is therefore read on the **administrative
axis** (`MUTATION_DATE`, tie-broken by `DATE_EFFECTIVE`), not the biographical one.
Rule of thumb for the whole pipeline: *`MUTATION_DATE` for administrative state,
`DATE_EFFECTIVE` for biographical chaining* (notebook 5).

A sensitivity check counts how many individuals would get a different terminal state
under the biographical ordering — reported for the data-quality section.

Net-of-suppression signals are computed per individual: a `Suppression*` cancels its
matching `Mutation*`, so a positive net count means the event stands.

In [3]:
CENSOR_DATE = pd.Period(clean["periode"].max(), freq="M").end_time
print("Observation window ends:", CENSOR_DATE.date())
print("\nCODETA_C states present:")
print(clean["CODETA_C"].value_counts(dropna=False).to_string())

# Administrative ordering; NaT sorted first so a dated row always wins tail(1).
ORDER_ADMIN = ["MUTATION_DATE", "DATE_EFFECTIVE"]
ev   = clean.sort_values(["id_projet"] + ORDER_ADMIN, na_position="first")
last = ev.groupby("id_projet").tail(1).set_index("id_projet")
term      = last["CODETA_C"]
term_code = last["CODE_MUTATION"]          # used for SuppressionDossier handling (§3)
natal     = last[["DATNAIS", "DATARR"]]    # native flag read on the same terminal row

# Sensitivity: terminal state under the biographical ordering (previous behaviour).
ev_bio   = clean.sort_values(["id_projet", "DATE_EFFECTIVE", "MUTATION_DATE"],
                             na_position="first")
term_bio = ev_bio.groupby("id_projet").tail(1).set_index("id_projet")["CODETA_C"]
n_diff   = (term != term_bio.reindex(term.index)).sum()
print(f"\nTerminal state differs between administrative and biographical ordering "
      f"for {n_diff:,} of {term.shape[0]:,} individuals "
      f"({n_diff / term.shape[0] * 100:.2f}%)")

def net_positive(code):
    supp = code.replace("Mutation", "Suppression", 1)
    pos  = clean["CODE_MUTATION"].eq(code).groupby(clean["id_projet"]).sum()
    neg  = clean["CODE_MUTATION"].eq(supp).groupby(clean["id_projet"]).sum()
    return pos.sub(neg, fill_value=0).clip(lower=0) > 0

has_nonconf  = net_positive("MutationDepartNonConfirme")
has_antic    = net_positive("MutationDepartAnticipe")
died_event   = net_positive("MutationDeces")
has_ddeces   = clean.groupby("id_projet")["DDECES"].apply(lambda s: s.notna().any())
antic_datdep = (clean.loc[clean["CODE_MUTATION"].eq("MutationDepartAnticipe")]
                     .groupby("id_projet")["DATDEP"].max())
print("\nSignals computed for", term.shape[0], "individuals")

Observation window ends: 2026-05-31

CODETA_C states present:
CODETA_C
Actif                483801
Inactif              234683
DepartAnticipe       121724
ArriveeProvisoire     23369

Terminal state differs between administrative and biographical ordering for 18,763 of 291,763 individuals (6.43%)

Signals computed for 291763 individuals


## 3. Binary departure target + native flag

Decision tree on the terminal state:

- Terminal event is **`SuppressionDossier`** → `dossier_supprime` (the whole file is
  administratively wiped — typically an erroneous registration; the example case is
  a same-day arrival/`DATDEM` fully reverted). These individuals carry
  `CODETA_C = Inactif` and would otherwise be misclassified as confirmed departures;
  they are **held out of the modelling population**, like deaths.
- `Inactif` **with** death → `death` (held out / censored)
- `Inactif` **without** death → `departed_confirmed`
- `DepartAnticipe` from a **non-confirmed** departure → `departed_nonconfirmed`
  (the move already happened; only inter-commune confirmation is pending → counts now)
- `DepartAnticipe` from an **anticipated** departure → `departed_anticipated_past`
  only once the announced `DATDEP` is reached; otherwise `pending_anticipated`
  (right-censored, `departed = 0`)
- `Actif` / `ArriveeProvisoire` → `present` (includes those temporarily out of commune)

`departed = 1` for any `departed_*`. A `ne_et_reside_depuis_naissance` flag marks
residents whose `DATARR` equals `DATNAIS` — i.e. living in Lausanne since birth. Both
dates are read on the **same terminal row** as the state (a `.first()` on the unsorted
frame would be file-order-dependent for returnees whose `DATARR` was reset).

In [4]:
idx = pd.Index(clean["id_projet"].dropna().unique(), name="id_projet")
ppl = pd.DataFrame(index=idx)
ppl["terminal_state"]    = term.reindex(idx)
ppl["dossier_supprime"]  = term_code.reindex(idx).eq("SuppressionDossier")
ppl["has_nonconf"]       = has_nonconf.reindex(idx, fill_value=False)
ppl["has_antic"]         = has_antic.reindex(idx, fill_value=False)
ppl["died"]              = (died_event.reindex(idx, fill_value=False)
                            | has_ddeces.reindex(idx, fill_value=False))
ppl["antic_datdep"]      = antic_datdep.reindex(idx)

def classify(r):
    if r["dossier_supprime"]:
        return "dossier_supprime"
    s = r["terminal_state"]
    if s == "Inactif":
        return "death" if r["died"] else "departed_confirmed"
    if s == "DepartAnticipe":
        if r["has_nonconf"]:
            return "departed_nonconfirmed"
        if r["has_antic"] and pd.notna(r["antic_datdep"]) and r["antic_datdep"] <= CENSOR_DATE:
            return "departed_anticipated_past"
        return "pending_anticipated"
    return "present"

ppl["depart_status"] = ppl.apply(classify, axis=1)
ppl["departed"]      = ppl["depart_status"].str.startswith("departed").astype(int)

# Native flag: DATARR == DATNAIS on the terminal row -> living in Lausanne since birth
ppl["ne_et_reside_depuis_naissance"] = (
    natal["DATNAIS"].dt.normalize().reindex(idx) == natal["DATARR"].dt.normalize().reindex(idx)
).fillna(False)

print("Departure status breakdown:")
print(ppl["depart_status"].value_counts().to_string())
print(f"\nTarget departed=1: {ppl['departed'].sum():,} ({ppl['departed'].mean()*100:.1f}%)")
print(f"Held out — death: {(ppl['depart_status']=='death').sum():,}  |  "
      f"dossier supprime: {(ppl['depart_status']=='dossier_supprime').sum():,}  |  "
      f"pending anticipated: {(ppl['depart_status']=='pending_anticipated').sum():,}")
print(f"Lausanne since birth: {ppl['ne_et_reside_depuis_naissance'].sum():,} "
      f"({ppl['ne_et_reside_depuis_naissance'].mean()*100:.1f}%)")

Departure status breakdown:
depart_status
departed_confirmed           166097
present                      111853
death                         11825
dossier_supprime               1407
departed_nonconfirmed           207
pending_anticipated             190
departed_anticipated_past       184

Target departed=1: 166,488 (57.1%)
Held out — death: 11,825  |  dossier supprime: 1,407  |  pending anticipated: 190
Lausanne since birth: 32,986 (11.3%)


## 4. Save & next steps

Two artifacts: `masterfile_events` (audited master + `famille_evenement`,
`nature_evenement`, `effet_taille_menage`) and `person_target` (one row per
individual: target, granular status incl. `dossier_supprime`, native flag).
Downstream notebooks must **exclude `depart_status ∈ {death, dossier_supprime}`**
from the modelling population. **Next (Phase 3):** reconstruct episodes per
`id_projet`, seed episode 0 from the 2014 snapshot, track household composition via
`effet_taille_menage` and `PERSMEN`, and aggregate to the person-level analytical
table carrying these targets.

In [5]:
def save(df, stem):
    pq = DATA_DIR / f"{stem}.parquet"
    try:
        df.to_parquet(pq, index=(df.index.name is not None))
        return pq
    except Exception as e:
        gz = pq.with_suffix(".csv.gz")
        df.to_csv(gz, index=(df.index.name is not None), compression="gzip")
        print(f"  {stem}: parquet unavailable ({type(e).__name__}) -> CSV gzip")
        return gz

p1 = save(clean, "masterfile_events")
p2 = save(ppl.reset_index(), "person_target")
print("Saved:", p1.name, "|", p2.name)

Saved: masterfile_events.parquet | person_target.parquet
